# Chapter 4.3 Sampling Depth

**TL;DR** — Isolates sampling-depth, WFR-FM sensitivity, stochastic bridge, and claim-boundary audit experiments for Chapter 4. The reader finishes with final Figure 4.11 panels, sampling-depth CSVs, WFR-FM source references, and the prior-boundary audit table.

**Prerequisites** - Chapter 4.1 (coupling geometry) and Chapter 4.2 (state-space assumptions).

**Outputs**
- `figures/ch04/new3/fig4_11a_raw_observed_counts.png`
- `figures/ch04/new3/fig4_11a_raw_observed_counts.pdf`
- `figures/ch04/new3/fig4_11a_raw_observed_counts.svg`
- `figures/ch04/new3/fig4_11b_equal_depth_composition.png`
- `figures/ch04/new3/fig4_11b_equal_depth_composition.pdf`
- `figures/ch04/new3/fig4_11b_equal_depth_composition.svg`
- `figures/ch04/new3/fig4_11c_sampling_depth_bootstrap_sensitivity.png`
- `figures/ch04/new3/fig4_11c_sampling_depth_bootstrap_sensitivity.pdf`
- `figures/ch04/new3/fig4_11c_sampling_depth_bootstrap_sensitivity.svg`
- `figures/ch04/new3/fig4_11d_wfrfm_raw_minus_equal_growth_heatmap.png`
- `figures/ch04/new3/fig4_11d_wfrfm_raw_minus_equal_growth_heatmap.pdf`
- `figures/ch04/new3/fig4_11d_wfrfm_raw_minus_equal_growth_heatmap.svg`
- `figures/ch04/new3/fig4_11e_wfrfm_mass_convention_agreement_summary.png`
- `figures/ch04/new3/fig4_11e_wfrfm_mass_convention_agreement_summary.pdf`
- `figures/ch04/new3/fig4_11e_wfrfm_mass_convention_agreement_summary.svg`
- `figures/ch04/new3/fig4_11f_stochastic_bridge_demo.png`
- `figures/ch04/new3/fig4_11f_stochastic_bridge_demo.pdf`
- `figures/ch04/new3/fig4_11f_stochastic_bridge_demo.svg`
- `outputs/ch04/table4_6_eb_downsampling_diagnostics.csv`
- `outputs/ch04/table4_6b_eb_bridge_sampling_diagnostics.csv`
- `outputs/ch04/table4_7_biological_assumption_boundary.csv`
- `outputs/ch04/tableA_4_3_prior_boundary_audit.csv`
- `outputs/ch04/cache/exp9_depth_sweep_growth_proxy.csv`
- `outputs/ch04/cache/exp9_adjacent_bridge_ot_sampled_endpoint_diagnostics.csv`

**Runtime** - Chapter 4.3 uses `CH04_SMOKE_MODE` rather than a separate `QUICK_MODE`; smoke mode reduces training-related defaults, NFE, and bridge sampling caps for CI. On CPU, most cells are diagnostics and plotting, with WFR-FM artifacts loaded from precomputed outputs when available.

**Key parameters**
- `DEFAULT_SEED = 42` and `SEEDS = [42, 43, 44]` come from `make_ch04_run_config()`.
- EB state space: standardized `PC-20`, with coarse bins derived from PC-20 because the EB file has no state/cluster labels.
- Defaults: `TRAINING_STEPS = 1500`, `TOY_TRAINING_STEPS = TRAINING_STEPS`, `BATCH_SIZE = 256`, `DEFAULT_NFE = 64`, and `NFE_GRID = [2, 4, 8, 16, 32, 64]`; smoke caps these.
- `bridge_cap = 120` in smoke mode and `800` otherwise.
- Final figure outputs use `figures/ch04/new3` and the `png/pdf/svg` format set.

## 0. Setup

This section makes the notebook independently runnable. It imports dependencies, resolves project paths, selects the device, fixes random seeds, creates output/cache directories, configures plotting, and exposes environment-variable overrides for smoke tests or controlled reruns.

Keep the defaults below as the chapter defaults: figures go to `figures/ch04`, tables and JSON summaries go to `outputs/ch04`, and intermediate caches go to `outputs/ch04/cache` unless `CH04_SMOKE_MODE=1` is set.

The setup is intentionally compact: one cell bootstraps imports and the project root, one cell declares run controls and output paths, and one cell imports the project helpers used below.


### Hypothesis
This setup assumes sampling-depth diagnostics can be run independently with the Chapter 4 defaults and bounded rerun controls.

### Setup
The cells define imports, `config`, `SEEDS`, `SOURCE_TIME`, `TARGET_TIME`, `TRAINING_STEPS`, `BATCH_SIZE`, `DEFAULT_NFE`, output directories, and cache paths. Project helpers for models, samplers, OT, metrics, plotting, and sampling-depth reports are imported for the later sections.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch04")

from pathlib import Path
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

try:
    import torch
    from torch import nn
except Exception as exc:
    raise ImportError("Chapter 4 experiments require PyTorch.") from exc

try:
    import anndata as ad
except Exception:
    ad = None

from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

import sys

ROOT_HINT = Path.cwd().resolve()
if not (ROOT_HINT / "src").is_dir():
    ROOT_HINT = ROOT_HINT.parent
if str(ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(ROOT_HINT))

from src.tutorial_init import apply_tutorial_plot_style, bootstrap, make_ch04_run_config, make_save_and_show
from src.utils import set_seed

boot = bootstrap(chapter="ch04", seed=42, quick_mode=None)
PROJECT_ROOT = boot.project_root
FIG_DIR = boot.fig_dir
OUT_DIR = boot.out_dir
DEVICE = boot.device

from src.core.models import VelocityMLP as ProjectVelocityMLP, count_parameters
from src.core.losses import cfm_batch, cfm_loss_from_pairs
from src.core.train import train_cfm_steps
from src.core.sampling import euler_sample
from src.data.samplers import CouplingPairSampler as SrcCouplingPairSampler
from src.core.ot import independent_coupling, coupling_diagnostics
from src.evaluation.metrics import (
    endpoint_metrics,
    fate_mass_error,
    coupling_l1_distance,
    normalized_cost_matrix,
    distribution_readout_metrics,
)
from src.evaluation.representations import (
    fit_pca_state_space,
    pca_inverse_transform,
    program_index_dict,
    readout_program_scores_from_matrix,
    standardize_train_space,
)


In [ ]:
config = make_ch04_run_config()
SEEDS = config.seeds
DEFAULT_SEED = config.default_seed
SOURCE_TIME = config.source_time
TARGET_TIME = config.target_time
TRAINING_STEPS = config.training_steps
BATCH_SIZE = config.batch_size
DEFAULT_NFE = config.default_nfe
NFE_GRID = config.nfe_grid
SINKHORN_EPSILON = config.sinkhorn_epsilon
EPSILON_GRID = config.epsilon_grid
SMOKE_MODE = config.smoke_mode
DEVICE = config.device
BOOTSTRAP_REPEATS = int(os.environ.get("CH04_BOOTSTRAP_REPEATS", "50"))
EB_MAX_CELLS_PER_TIME = os.environ.get("CH04_EB_MAX_CELLS_PER_TIME", "")
EB_MAX_CELLS_PER_TIME = None if EB_MAX_CELLS_PER_TIME == "" else int(EB_MAX_CELLS_PER_TIME)
TOY_TRAINING_STEPS = int(os.environ.get("CH04_TOY_TRAINING_STEPS", str(TRAINING_STEPS)))
if SMOKE_MODE:
    TOY_TRAINING_STEPS = min(TOY_TRAINING_STEPS, 20)
    BOOTSTRAP_REPEATS = min(BOOTSTRAP_REPEATS, 3)
    EB_MAX_CELLS_PER_TIME = 120 if EB_MAX_CELLS_PER_TIME is None else min(EB_MAX_CELLS_PER_TIME, 120)

DATA_DIR = PROJECT_ROOT / "data"
EB_PATH = DATA_DIR / "trajectorynet_eb" / "eb_velocity_v5.npz"
TOY_DIR = DATA_DIR / "toy_branching_snapshots"
TOY_CSV_PATH = TOY_DIR / "observed_2d_snapshots.csv"
TOY_H5AD_PATH = TOY_DIR / "branching_toy_pseudocounts.h5ad"

FIG_DIR = PROJECT_ROOT /  "figures" / "ch04"
OUT_DIR = PROJECT_ROOT /  "outputs" / "ch04"
if SMOKE_MODE:
    FIG_DIR = FIG_DIR / "smoke"
    OUT_DIR = OUT_DIR / "smoke"
CACHE_DIR = OUT_DIR / "cache"
FINAL_FIG_DIR = PROJECT_ROOT / "figures" / "ch04" / "new3"
for path in [FIG_DIR, OUT_DIR, CACHE_DIR, FINAL_FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

apply_tutorial_plot_style()
PALETTE = {
    "source": "#4C78A8",
    "target": "#F58518",
    "random": "#8E8E8E",
    "ot": "#008A7A",
    "reflow1": "#5369A6",
    "reflow2": "#B279A2",
    "rare": "#D95F02",
    "major": "#2C7FB8",
    "program": "#54A24B",
    "diagnostic": "#E45756",
}

print(
    f"Chapter 4.3 setup: device={DEVICE}, seed={DEFAULT_SEED}, "
    f"steps={TRAINING_STEPS}, toy_steps={TOY_TRAINING_STEPS}, "
    f"batch={BATCH_SIZE}, default_nfe={DEFAULT_NFE}, smoke={SMOKE_MODE}"
)


In [ ]:

from src.core.models import VelocityMLP as ProjectVelocityMLP, count_parameters
from src.data.samplers import CouplingPairSampler as SrcCouplingPairSampler
from src.core.ot import independent_coupling, coupling_diagnostics
from src.evaluation.metrics import (
    endpoint_metrics,
    fate_mass_error,
    coupling_l1_distance,
    normalized_cost_matrix,
    distribution_readout_metrics,
)
from src.evaluation.representations import (
    fit_pca_state_space,
    pca_inverse_transform,
    program_index_dict,
    readout_program_scores_from_matrix,
    standardize_train_space,
)

from src.artifacts import json_ready, load_json, load_npz, load_pt as _load_pt, save_csv, save_json, save_npz, save_pt
from src.artifacts import artifact_exists, as_float32, ensure_finite, sample_rows, stable_hash, to_tensor as _to_tensor


def load_pt(path: str | Path, map_location=None):
    return _load_pt(path, map_location=map_location or DEVICE)


def to_tensor(x, device: torch.device = DEVICE):
    return _to_tensor(x, device)

from src.visualization.sampling_depth import (
    bridge_sampling_diagnostic,
    display_ch04_png,
    display_final_png as _display_final_png,
    load_eb_data as _load_eb_data,
    load_wfrfm_sampling_outputs,
    make_wfrfm_agreement_summary,
    make_wfrfm_growth_delta_grid,
    plot_equal_depth_composition,
    plot_raw_observed_counts,
    plot_sampling_depth_bootstrap_sensitivity,
    plot_stochastic_bridge_demo,
    plot_wfrfm_agreement_summary,
    plot_wfrfm_growth_delta_heatmap,
    remember_source as _remember_source,
    resolve_required_artifact as _resolve_required_artifact,
    safe_relpath as _safe_relpath,
    save_figure as _save_figure,
    save_pub_figure as _save_pub_figure,
    write_claim_boundary_checklist,
    write_final_figure_package,
)

FINAL_SOURCE_PATHS: dict[str, str] = {}

save_figure = lambda fig, filename, close=True: _save_figure(fig, FIG_DIR, filename, close=close)
save_pub_figure = lambda fig, stem, close=True: _save_pub_figure(fig, FINAL_FIG_DIR, stem, close=close)
display_saved_figure = lambda filename, width=900: display_ch04_png((FIG_DIR / filename).parent, (FIG_DIR / filename).name, width=width) if not Path(filename).is_absolute() else display_ch04_png(Path(filename).parent, Path(filename).name, width=width)
display_final_png = lambda stem: _display_final_png(FINAL_FIG_DIR, stem)
safe_relpath = lambda path, root=PROJECT_ROOT: _safe_relpath(path, root)
resolve_required_artifact = lambda filename, preferred_dirs=None: _resolve_required_artifact(filename, preferred_dirs=preferred_dirs or [], search_root=PROJECT_ROOT)
remember_source = lambda name, path: _remember_source(FINAL_SOURCE_PATHS, name, path, root=PROJECT_ROOT)

save_and_show = make_save_and_show(FIG_DIR, write_pdf=False, save_fn=save_figure)


## 1. Shared Utilities

The utility cells define the shared tutorial machinery used by the EB and toy sections: artifact writers, model wrappers, training/loading helpers, Sinkhorn diagnostics, rollout metrics, and plotting helpers.

The intent is auditability. The key CFM/Sinkhorn/rollout/metric logic stays visible in the notebook, while tested primitives from `src` are reused where available.

The shared utility cell imports the small set of runtime, metric, and OT helpers used below. This split does not train a new EB model; it audits sampling-depth and claim-boundary behavior around already-defined Chapter 4 objects.


### Hypothesis
The sampling-depth notebook assumes shared metric and OT utilities are sufficient for auditing depth and claim-boundary behavior without training a new EB model.

### Setup
The utility cells import `mmd_rbf`, `_sliced_w2`, `compute_cost_matrix`, `sample_from_plan`, `sinkhorn_plan`, and `coupling_diagnostics`. The local `sliced_w2` wrapper fixes the projection-budget interface used by downstream metric summaries.


In [ ]:
from src.evaluation.metrics import mmd_rbf, sliced_w2 as _sliced_w2
from src.core.ot import compute_cost_matrix, sample_from_plan, sinkhorn_plan
from src.core.ot import coupling_diagnostics


def sliced_w2(X, Y, n_projections: int = 128, seed: int = DEFAULT_SEED):
    return _sliced_w2(X, Y, n_projections=n_projections, seed=seed)


## 2. Load EB Data

This section establishes the data contract for all EB experiments in this tutorial.

- Input: `data/eb_velocity_v5.npz`.
- Training and distributional metrics: standardized PC-20, built from `pcs[:, :20]` even though the file stores 100 PCs.
- Display only: two-dimensional `phate` coordinates.
- Main bridge: EB time label `1 -> 2`.
- Artifact: `outputs/ch04/eb_data_summary.json`, which records shapes, labels, standardization, and claim-boundary notes.

Load the EB snapshot once, then inspect snapshot counts before the sampling-depth experiments begin. The loader is presentation infrastructure; the diagnostics below define the tutorial logic.


### Hypothesis
The EB sampling-depth diagnostics assume a single standardized PC-space data contract and display-only PHATE coordinates.

### Setup
`_load_eb_data` returns the EB payload, snapshot counts, standardized PC endpoints, and PHATE display endpoints. `fit_pc_to_phate_mapper` defines `pc_to_phate`, and `off_manifold_reference_pc` records the all-snapshot PC-space reference.


In [ ]:
EB = _load_eb_data(
    EB_PATH,
    source_time=SOURCE_TIME,
    target_time=TARGET_TIME,
    out_dir=OUT_DIR,
    max_cells_per_time=EB_MAX_CELLS_PER_TIME,
    seed=DEFAULT_SEED,
)
EB["counts"]


In [ ]:
from src.data.loading import fit_pc_to_phate_mapper

pc_to_phate = fit_pc_to_phate_mapper(EB["pcs20_all"], EB["phate_all"], n_neighbors=15)

X0_eb, X1_eb = EB["X0_pc"], EB["X1_pc"]
X0p_eb, X1p_eb = EB["X0_phate"], EB["X1_phate"]
off_manifold_reference_pc = EB["pcs20_all"]
off_manifold_reference_note = "all available EB snapshots in standardized PC-20"
print(off_manifold_reference_note, off_manifold_reference_pc.shape)


## Exp 9. EB Equal-Depth Subsampling

Tutorial goal: separate sampling-depth effects from calibrated biological abundance claims.

This experiment uses all EB time labels. Because the EB snapshots are destructive samples, raw observed cell counts are treated as sampling-depth proxies. If no observed state labels are present, the notebook creates coarse PC-20 state bins only for composition diagnostics. Equal-depth subsampling changes the mass convention to the same sampled depth per time point; it does not establish calibrated abundance, expansion, loss, or state uncertainty.

### Hypothesis
This experiment tests whether changing the sampling-depth convention alters EB composition and growth-proxy diagnostics without implying calibrated abundance.

### Setup
The cells derive `labels_all`, `pcs_all`, `unique_times`, `state_bins`, and `counts_by_state`, then form `equal_counts` by equal-depth subsampling. `depth_grid`, bootstrap loops, and `downsampling_table` summarize raw versus equal-depth growth-proxy behavior.


Start Exp 9 by building coarse diagnostic state bins in standardized PC-20. These are audit bins because the EB file does not provide validated cell-state labels.

In [ ]:
labels_all = EB["labels"]
pcs_all = EB["pcs20_all"]
unique_times = sorted(np.unique(labels_all).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
# EB npz has no cell-type/state labels; create coarse bins on PC-20.
n_bins = min(8, max(2, int(np.sqrt(len(unique_times) * 8))))
kmeans = KMeans(n_clusters=n_bins, random_state=42, n_init=10)
state_bins = kmeans.fit_predict(pcs_all).astype(str)
all_bins = sorted(np.unique(state_bins).tolist(), key=lambda s: int(s) if str(s).isdigit() else s)

Next compare the raw destructive snapshot depth with an equal-depth subsample at each time point. The final paper-ready outputs are two independent small figures: raw observed counts and equal-depth composition, both using the same state-bin colors.

In [ ]:
counts_rows = []
for t in unique_times:
    mask = labels_all == t
    total = int(mask.sum())
    count_by_bin = pd.Series(state_bins[mask]).value_counts()
    for b in all_bins:
        c = int(count_by_bin.get(b, 0))
        counts_rows.append({"time": t, "state_bin": b, "n_cells": c, "total_time_cells": total, "proportion": float(c / total)})
counts_by_state = pd.DataFrame(counts_rows)

n_min = int(pd.Series(labels_all).value_counts().min())
rng = np.random.default_rng(303)
equal_selected = []
for t in unique_times:
    idx = np.flatnonzero(labels_all == t)
    equal_selected.append(rng.choice(idx, size=n_min, replace=False))
equal_selected = np.concatenate(equal_selected)
equal_rows = []
for t in unique_times:
    idx_t = equal_selected[labels_all[equal_selected] == t]
    count_by_bin = pd.Series(state_bins[idx_t]).value_counts()
    for b in all_bins:
        equal_rows.append({"time": t, "state_bin": b, "n_cells": int(count_by_bin.get(b, 0)), "total_time_cells": int(len(idx_t))})
equal_counts = pd.DataFrame(equal_rows)

In [ ]:
_raw_counts_path = plot_raw_observed_counts(
    counts_by_state,
    unique_times=unique_times,
    all_bins=all_bins,
    final_fig_dir=FINAL_FIG_DIR,
)
_equal_depth_composition_path = plot_equal_depth_composition(
    equal_counts,
    unique_times=unique_times,
    all_bins=all_bins,
    n_min=n_min,
    final_fig_dir=FINAL_FIG_DIR,
)


The bootstrap sweep measures how much the log growth proxy changes when each time point is forced to the same sampling depth.

In [ ]:
depth_grid = [d for d in [100, 200, 400, n_min] if d <= n_min]
depth_grid = sorted(set(depth_grid))
bootstrap_count_rows = []
for depth in depth_grid:
    for rep in range(BOOTSTRAP_REPEATS):
        selected = []
        for t in unique_times:
            idx = np.flatnonzero(labels_all == t)
            selected.append(rng.choice(idx, size=depth, replace=False))
        selected = np.concatenate(selected)
        for t in unique_times:
            idx_t = selected[labels_all[selected] == t]
            count_by_bin = pd.Series(state_bins[idx_t]).value_counts()
            for b in all_bins:
                c = int(count_by_bin.get(b, 0))
                bootstrap_count_rows.append({
                    "depth": int(depth), "repeat": int(rep), "time": t, "state_bin": b,
                    "count": c, "proportion": float(c / int(depth)),
                })
boot_counts = pd.DataFrame(bootstrap_count_rows)

In [ ]:
eps_growth = 0.5
raw_rows = []
for t0, t1 in zip(unique_times[:-1], unique_times[1:]):
    c0 = counts_by_state[counts_by_state["time"] == t0].set_index("state_bin")
    c1 = counts_by_state[counts_by_state["time"] == t1].set_index("state_bin")
    for b in all_bins:
        n0 = int(c0.loc[b, "n_cells"])
        n1 = int(c1.loc[b, "n_cells"])
        p0 = float(c0.loc[b, "proportion"])
        p1 = float(c1.loc[b, "proportion"])
        raw_rows.append({
            "time_bridge": f"{t0}->{t1}",
            "time_t": t0,
            "time_t_next": t1,
            "state_bin": b,
            "raw_count_t": n0,
            "raw_count_t_next": n1,
            "raw_count_growth_proxy": float(np.log((n1 + eps_growth) / (n0 + eps_growth))),
            "normalized_proportion_change": float(p1 - p0),
        })
raw_growth = pd.DataFrame(raw_rows)

boot_proxy_rows = []
for depth in depth_grid:
    for rep in range(BOOTSTRAP_REPEATS):
        sub = boot_counts[(boot_counts["depth"] == depth) & (boot_counts["repeat"] == rep)]
        for t0, t1 in zip(unique_times[:-1], unique_times[1:]):
            s0 = sub[sub["time"] == t0].set_index("state_bin")
            s1 = sub[sub["time"] == t1].set_index("state_bin")
            for b in all_bins:
                n0 = int(s0.loc[b, "count"])
                n1 = int(s1.loc[b, "count"])
                boot_proxy_rows.append({
                    "depth": int(depth),
                    "repeat": int(rep),
                    "time_bridge": f"{t0}->{t1}",
                    "state_bin": b,
                    "equal_depth_growth_proxy": float(np.log((n1 + eps_growth) / (n0 + eps_growth))),
                })
boot_proxy = pd.DataFrame(boot_proxy_rows)
proxy_summary_all_depths = boot_proxy.groupby(["depth", "time_bridge", "state_bin"]).agg(
    proxy_mean=("equal_depth_growth_proxy", "mean"),
    proxy_ci_low=("equal_depth_growth_proxy", lambda x: float(np.quantile(x, 0.025))),
    proxy_ci_high=("equal_depth_growth_proxy", lambda x: float(np.quantile(x, 0.975))),
).reset_index()
proxy_equal_depth = proxy_summary_all_depths[proxy_summary_all_depths["depth"] == n_min].rename(columns={
    "proxy_mean": "equal_depth_proxy_mean",
    "proxy_ci_low": "equal_depth_proxy_ci_low",
    "proxy_ci_high": "equal_depth_proxy_ci_high",
})

In [ ]:
downsampling_table = raw_growth.merge(
    proxy_equal_depth[["time_bridge", "state_bin", "equal_depth_proxy_mean", "equal_depth_proxy_ci_low", "equal_depth_proxy_ci_high"]],
    on=["time_bridge", "state_bin"],
    how="left",
)
ci_width = downsampling_table["equal_depth_proxy_ci_high"] - downsampling_table["equal_depth_proxy_ci_low"]
inside_ci = (downsampling_table["raw_count_growth_proxy"] >= downsampling_table["equal_depth_proxy_ci_low"]) & (downsampling_table["raw_count_growth_proxy"] <= downsampling_table["equal_depth_proxy_ci_high"])
downsampling_table["stable_under_subsampling"] = np.where(inside_ci & (ci_width < 1.0), "stable", "sensitive")
downsampling_table["state_label_source"] = "k-means on standardized PC-20 because EB file has no state/cluster labels"
downsampling_table["abundance_claim_boundary"] = "raw counts reflect sampling depth; not absolute biological abundance"
_ = save_csv(OUT_DIR / "table4_6_eb_downsampling_diagnostics.csv", downsampling_table)
_ = save_csv(CACHE_DIR / "exp9_depth_sweep_growth_proxy.csv", proxy_summary_all_depths)
downsampling_table.head()


The adjacent-bridge diagnostic samples endpoints from balanced OT plans. It remains a PC-20 transport diagnostic, not a trained CFM bridge or a growth model.

In [ ]:
bridge_cap = 120 if SMOKE_MODE else 800
bridge_table_path = OUT_DIR / "table4_6b_eb_bridge_sampling_diagnostics.csv"
force_recompute_bridge = os.environ.get("CH04_FORCE_RECOMPUTE_BRIDGE_DIAGNOSTICS", "0") == "1"

if bridge_table_path.exists() and bridge_table_path.stat().st_size > 0 and not force_recompute_bridge:
    bridge_sampling_table = pd.read_csv(bridge_table_path)
else:
    bridge_rows = []
    for bridge_id, (a, b) in enumerate(zip(unique_times[:-1], unique_times[1:])):
        bridge_rows.append(bridge_sampling_diagnostic(
            pcs_all=pcs_all,
            labels_all=labels_all,
            state_bins=state_bins,
            all_bins=all_bins,
            time_a=a,
            time_b=b,
            sampling_setting="original_depth",
            cap=bridge_cap,
            seed=410 + 10 * bridge_id,
            sinkhorn_epsilon=SINKHORN_EPSILON,
        ))
        bridge_rows.append(bridge_sampling_diagnostic(
            pcs_all=pcs_all,
            labels_all=labels_all,
            state_bins=state_bins,
            all_bins=all_bins,
            time_a=a,
            time_b=b,
            sampling_setting="equal_depth",
            cap=min(bridge_cap, n_min),
            seed=411 + 10 * bridge_id,
            sinkhorn_epsilon=SINKHORN_EPSILON,
        ))
    bridge_sampling_table = pd.DataFrame(bridge_rows)
    _ = save_csv(bridge_table_path, bridge_sampling_table)
_ = save_csv(CACHE_DIR / "exp9_adjacent_bridge_ot_sampled_endpoint_diagnostics.csv", bridge_sampling_table)
_ = remember_source("table4_6b_eb_bridge_sampling_diagnostics.csv", bridge_table_path)
bridge_sampling_table

In [ ]:
boundary_rows = [
    {"assumption": "state labels", "status": "coarse k-means bins used", "claim_boundary": "diagnostic bins, not validated cell types"},
    {"assumption": "cell counts", "status": "observed destructive snapshot counts", "claim_boundary": "sampling depth, not absolute biological abundance"},
    {"assumption": "subsampling", "status": f"equal-depth bootstrap repeats={BOOTSTRAP_REPEATS}", "claim_boundary": "stability diagnostic only"},
    {"assumption": "adjacent bridge OT", "status": "original-depth and equal-depth OT sampled endpoint diagnostics in PC-20", "claim_boundary": "diagnostic only; not trained CFM and not observed paired histories"},
]
boundary_table = pd.DataFrame(boundary_rows)
_ = save_csv(OUT_DIR / "table4_7_biological_assumption_boundary.csv", boundary_table)
boundary_table

Finally, plot raw-count growth proxies against equal-depth bootstrap intervals. This is a sampling-depth diagnostic and not an abundance calibration: the visual emphasizes which raw proxies fall outside the equal-depth 95% intervals.

In [ ]:
downsampling_plot_path = remember_source(
    "table4_6_eb_downsampling_diagnostics.csv",
    resolve_required_artifact("table4_6_eb_downsampling_diagnostics.csv", [OUT_DIR]),
)
downsampling_table = pd.read_csv(downsampling_plot_path)
plot_df = plot_sampling_depth_bootstrap_sensitivity(downsampling_table, final_fig_dir=FINAL_FIG_DIR)
plot_df.head()


## Exp 9b. WFR-FM Sampling-Depth Sensitivity

Tutorial goal: load the internal WFR-FM sampling-depth sensitivity outputs and audit how the growth readout changes under raw observed depth versus equal-depth control.

The WFR-FM outputs are read as a mass-convention sensitivity diagnostic. They are not calibrated biological abundance measurements. The final small figures below show only the raw-minus-equal growth heatmap and a compact agreement summary.

### Hypothesis
This diagnostic tests whether WFR-FM growth readouts depend on the raw-depth versus equal-depth mass convention.

### Setup
`load_wfrfm_sampling_outputs` loads `wfrfm_growth_by_bin`, `wfrfm_sampling_sensitivity`, and `wfrfm_summary`. `make_wfrfm_growth_delta_grid`, `plot_wfrfm_growth_delta_heatmap`, and `make_wfrfm_agreement_summary` generate the sensitivity summaries.


In [ ]:
wfrfm_outputs = load_wfrfm_sampling_outputs(
    out_dir=OUT_DIR,
    project_root=PROJECT_ROOT,
    source_paths=FINAL_SOURCE_PATHS,
)
wfrfm_growth_by_bin = wfrfm_outputs.growth_by_bin
wfrfm_sampling_sensitivity = wfrfm_outputs.sampling_sensitivity
wfrfm_summary = wfrfm_outputs.summary

wfrfm_delta = make_wfrfm_growth_delta_grid(wfrfm_growth_by_bin)
plot_wfrfm_growth_delta_heatmap(wfrfm_delta, final_fig_dir=FINAL_FIG_DIR)
wfrfm_agreement_summary = make_wfrfm_agreement_summary(wfrfm_sampling_sensitivity)
plot_wfrfm_agreement_summary(wfrfm_agreement_summary, final_fig_dir=FINAL_FIG_DIR)

print("WFR-FM source summary:")
print({
    "implementation": wfrfm_summary.get("implementation"),
    "output_suffix": wfrfm_summary.get("output_suffix"),
    "epochs": wfrfm_summary.get("epochs"),
    "equal_seeds": wfrfm_summary.get("equal_seeds"),
    "raw_observed_depth_was_capped": wfrfm_summary.get("raw_observed_depth_was_capped"),
})
display(wfrfm_agreement_summary)


## Exp 10. Stochastic Bridge Demo

Tutorial goal: show that stochastic bridge width is a separate path-family assumption from the EB sampling-depth and WFR-FM mass-convention diagnostics.

This is a synthetic normalized demo. The point clouds are not EB observations and are not evidence for calibrated abundance, growth, or state uncertainty.

### Hypothesis
This demo tests the assumption that stochastic bridge width is a separate path-family choice from EB sampling-depth and mass-convention diagnostics.

### Setup
The section calls `plot_stochastic_bridge_demo` and records `t_grid_demo` for the normalized synthetic bridge. The output is a path-family illustration only, using the final figure directory rather than EB metric tables.


In [ ]:
_, t_grid_demo = plot_stochastic_bridge_demo(final_fig_dir=FINAL_FIG_DIR)


## Exp 11. Prior Boundary Audit

Tutorial goal: make explicit which biological priors can be used as diagnostics and which claims would require external evidence.

The next cell writes `tableA_4_3_prior_boundary_audit.csv`. This is a table-only prior audit; it is not a real-data prior experiment.

Claim boundary: any prior would require external evidence before upgrading a diagnostic into a biological claim.


### Hypothesis
This audit tests which biological-prior statements are diagnostics versus claims requiring external evidence.

### Setup
The cell builds `prior_rows` and `prior_audit` with prior type, injection point, external evidence, allowed claim, and forbidden claim fields. `save_csv` writes the boundary-audit table for manuscript-facing review.


In [ ]:
prior_rows = [
    {
        "prior": "RNA velocity alignment",
        "prior_type": "directional transcriptomic prior",
        "injection_point": "loss / diagnostic",
        "external_evidence_required": "validated velocity preprocessing, model assumptions, perturbation or barcoded-history corroboration",
        "allowed_claim": "flow is more aligned with a specified RNA-velocity vector field under stated preprocessing",
        "forbidden_claim": "learned coupling is observed same-cell history or validated developmental outcome",
    },
    {
        "prior": "marker monotonicity",
        "prior_type": "marker trend constraint",
        "injection_point": "loss / readout / diagnostic",
        "external_evidence_required": "curated marker set and expected direction from independent biology",
        "allowed_claim": "generated trajectories respect the specified marker monotonicity prior",
        "forbidden_claim": "marker prior proves true temporal ordering without external validation",
    },
    {
        "prior": "cell-type labels",
        "prior_type": "categorical annotation prior",
        "injection_point": "sampling / readout / diagnostic",
        "external_evidence_required": "validated annotation protocol and uncertainty handling",
        "allowed_claim": "terminal label proportions or forbidden transitions under provided labels",
        "forbidden_claim": "labels create same-cell histories",
    },
    {
        "prior": "barcoded history assays",
        "prior_type": "external paired-history evidence",
        "injection_point": "loss / sampling / evaluation",
        "external_evidence_required": "barcode assay with group calling and sampling bias model",
        "allowed_claim": "agreement with barcode-derived constraints",
        "forbidden_claim": "unobserved transitions are recovered without barcode coverage",
    },
    {
        "prior": "GRN constraints",
        "prior_type": "mechanistic regulatory prior",
        "injection_point": "loss / diagnostic",
        "external_evidence_required": "GRN source, confidence scores, cell-context validation",
        "allowed_claim": "flow is consistent with selected GRN constraints under the chosen model",
        "forbidden_claim": "GRN-constrained flow proves causal regulation by itself",
    },
]
prior_audit = pd.DataFrame(prior_rows)
_ = save_csv(OUT_DIR / "tableA_4_3_prior_boundary_audit.csv", prior_audit)
prior_audit


## Take-aways

- *Finding 1* — Equal-depth EB subsampling separates sampling-depth effects from calibrated biological abundance, expansion, loss, or state-uncertainty claims.
- *Finding 2* — WFR-FM outputs are read as mass-convention sensitivity diagnostics rather than calibrated biological abundance measurements.
- *Finding 3* — The stochastic bridge and prior-boundary sections mark interpretation boundaries, including when external evidence is required before a diagnostic becomes a biological claim.

Next: → ch5_1
